# Base Server

> The core abstraction for different FL Servers. Any general Server functionality resides here.

In [ ]:
#| default_exp servers.base_server

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

In [ ]:
#| export
from fastcore.utils import *
from fastcore.all import *
import os
import math
import pickle
import json
import gc

from collections import defaultdict, OrderedDict
from copy import deepcopy
import random
from enum import Enum

from tqdm import tqdm
from loguru import logger

import networkx as nx
from community import community_louvain

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from fedai.core import AgentRole
from fedai.utils import *
from fedai.client_selector import *
from fedai.optimizers import *
from fedai.utils import *
from fedai.metrics import *
from fedai.losses import *
from fedai.data import prepare_dl
from fedai.models import create_model
from fedai.optimizers import get_optimizer

In [ ]:
#| export
import torch

class StateManager:
    def __init__(self):
        self.registry = {} 

    def set_state(self, client_id, state_dict):
        """
        Takes a dictionary of tensors/objects and moves them to CPU.
        Works for weights, optimizer states, anchors (h), etc.
        """
        if client_id not in self.registry:
            self.registry[client_id] = {}

        for key, value in state_dict.items():
            # Deep clone and move to CPU if it's a torch object
            if isinstance(value, torch.Tensor):
                self.registry[client_id][key] = value.detach().cpu().clone()
            elif isinstance(value, dict):
                # Handle nested dicts (like optimizer.state_dict())
                self.registry[client_id][key] = self._to_cpu(value)
            else:
                self.registry[client_id][key] = value

    def get_state(self, client_id):       
        if client_id not in self.registered_ids:
            init_state = {}
            return init_state
     
        return self.registry.get(client_id, {})

    def _to_cpu(self, obj):
        """Recursively moves tensors in a nested structure to CPU."""
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().clone()
        elif isinstance(obj, dict):
            return {k: self._to_cpu(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._to_cpu(v) for v in obj]
        return obj

    @property
    def registered_ids(self):
        return list(self.registry.keys())

In [ ]:
#| hide
from fedai.models import create_model
from fedai.optimizers import get_optimizer
from omegaconf import OmegaConf
from fedai.client_selector import BaseClientSelector
import importlib


In [ ]:
#| hide
st_mgr = StateManager()
cfg = OmegaConf.load('./examples/cfg.yaml')
model = create_model(cfg)#("lenet_mnist")
opt = get_optimizer(cfg)(model.parameters(), lr= 0.001)

In [ ]:
#| hide
st_mgr.set_state(1, {'model': model.state_dict(), 'optimizer': opt.state_dict()})

In [ ]:
#| hide
st_mgr.get_state(1)

{'model': {'conv1.0.weight': tensor([[[[ 0.1529,  0.1660, -0.0469,  0.1837, -0.0438],
            [ 0.0404, -0.0974,  0.1175,  0.1763, -0.1467],
            [ 0.1738,  0.0374,  0.1478,  0.0271,  0.0964],
            [-0.0282,  0.1542,  0.0296, -0.0934,  0.0510],
            [-0.0921, -0.0235, -0.0812,  0.1327, -0.1579]]],
  
  
          [[[-0.0922, -0.0565, -0.1203,  0.0189, -0.1975],
            [ 0.1806, -0.1699,  0.1544,  0.0333, -0.0649],
            [ 0.1236,  0.0312,  0.1616,  0.0219, -0.0631],
            [ 0.0537, -0.0542,  0.0842,  0.1786,  0.1156],
            [-0.0874,  0.1155,  0.0358,  0.1016, -0.1219]]],
  
  
          [[[-0.1980, -0.0773, -0.1534,  0.1641,  0.0576],
            [ 0.0828,  0.0633, -0.0035,  0.1565, -0.1421],
            [ 0.0126, -0.1365,  0.0617, -0.0689,  0.0613],
            [-0.0417,  0.1659, -0.1185, -0.1193, -0.1193],
            [ 0.1799,  0.0667,  0.1925, -0.1651, -0.1984]]],
  
  
          [[[-0.1565, -0.1345,  0.0810,  0.0716,  0.1662],
     

In [ ]:
#| hide
opt.load_state_dict(st_mgr.get_state(1)['optimizer'])

In [ ]:
#| hide
st_mgr.get_state(2)

{}

In [ ]:
#| export
import torch

class DiskStateManager:
    def __init__(self, save_dir):
        self.save_dir = save_dir
        self.registry = {} 

    def set_state(self, client_id, state_dict):
        """
        Takes a dictionary of tensors/objects and moves them to CPU.
        Works for weights, optimizer states, anchors (h), etc.
        """
        if client_id not in self.registry:
            self.registry[client_id] = {}

        for key, value in state_dict.items():
            # Deep clone and move to CPU if it's a torch object
            if isinstance(value, torch.Tensor):
                self.registry[client_id][key] = value.detach().cpu().clone()
            elif isinstance(value, dict):
                # Handle nested dicts (like optimizer.state_dict())
                self.registry[client_id][key] = self._to_cpu(value)
            else:
                self.registry[client_id][key] = value

    def get_state(self, client_id):
        return self.registry.get(client_id, {})

    def _to_cpu(self, obj):
        """Recursively moves tensors in a nested structure to CPU."""
        if isinstance(obj, torch.Tensor):
            return obj.detach().cpu().clone()
        elif isinstance(obj, dict):
            return {k: self._to_cpu(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [self._to_cpu(v) for v in obj]
        return obj

## BaseServer

This class implements basic components of the Server abstraction and is considered the base class for all type of Servers

In [ ]:
#| export
class BaseServer:
    def __init__(self,
                 cfg,
                 client_selector,
                 client_cls,
                 criterion,
                 fds,
                 writer= None,
                 device= None,
                 **kwargs
                 ):  
                 
        self.cfg = cfg 
        self.client_selector = client_selector
        self.client_cls = client_cls
        self.criterion = criterion
        self.writer = writer
        self.fds = fds
        self.device = device
        self.model = create_model(self.cfg)
        for key, value in kwargs.items():
            setattr(self, key, value)
        self.state_mgr = StateManager()

We need to make sure that all clients start from the same initial model at round 0. So, we need to save the initial model once at the server side and load it at each client during its initialization.

In [ ]:
#| hide
from fedai.data import init_data, IMG_DATA_CONFIGS


In [ ]:
#| hide
cfg.data.partitioner.kwargs = {"alpha": 0.5, "partition_by": IMG_DATA_CONFIGS[cfg.data.name].y}
fds = init_data(cfg)

In [ ]:
#| hide
client_cls = importlib.import_module(f"fedai.clients.base_client")
client_cls= getattr(client_cls, "BaseClient")
client_selector = BaseClientSelector(cfg)
server = BaseServer(cfg= cfg,
                    client_selector= client_selector,
                    client_cls= client_cls,
                    criterion= nn.CrossEntropyLoss(),
                    fds= fds,
                    kwargs= {"anything": "value"}
                    )


In [ ]:
#| export
@patch
def client_fn(self: BaseServer, id, comm_round, client_state):

    if comm_round == 1 and client_state == {}:
        client_state['model'] = self.model.state_dict()

    model = create_model(self.cfg)
    model.load_state_dict(client_state['model'])
    client_state['model'] = model

    optimizer = get_optimizer(self.cfg)(model.parameters(), lr= 0.001)#self.cfg.optimizer.lr)
    optimizer.load_state_dict(client_state['optimizer']) if 'optimizer' in client_state else None
    client_state['optimizer'] = optimizer

    train_loader = prepare_dl(self.cfg, id, self.fds, train=True, distributed=False)
    test_loader = prepare_dl(self.cfg, id, self.fds, train=False, distributed=False)
    client = self.client_cls(id= id, 
                             cfg= self.cfg,
                             train_loader= train_loader,
                             test_loader= test_loader,
                             state= client_state,
                             criterion= self.criterion,
                             device= self.device,
                             t= comm_round)
    return client


In [ ]:
#| hide
client = server.client_fn(id= 0,
                          comm_round= 1,
                          client_state= st_mgr.get_state(0),
                          )

In [ ]:
#| export
@patch
def train(self: BaseServer):
    res =  []
    selected_clients = self.client_selector.select()
    
    for t in range(1, self.cfg.n_rounds + 1):
        lst_active_ids = selected_clients[t-1]
        len_clients_ds = {}

        for id in lst_active_ids:
            client_state = self.state_mgr.get_state(id)
            client = self.client_fn(id= id, comm_round= t, client_state= client_state)
            client.fit()
            logger.info(f"Client {id} finished training.")
            logger.info("*"*20)
            self.state_mgr.set_state(id, client.save_state())
            
            len_clients_ds[id] = len(client.train_loader.dataset)

            del client 
            gc.collect()
            torch.cuda.empty_cache()

        self.aggregate(lst_active_ids, t, len_clients_ds)
        train_res, test_res = self.evaluate(t)
        # self.state_mgr.registry.clear() if self.cfg.agg == "one_model" else None
        
        train_df, test_df = self.writer.write(lst_active_ids, train_res, test_res, t) 
        res.append((train_df, test_df))
        
    self.writer.save(res)
    self.writer.finish()

    return res

In [ ]:
#| hide
# server.train()

Each subclass, should implement the `aggregate` method, which is responsible for aggregating the local models from the clients and updating the global model/local personalized models accordingly.

In [ ]:
#| export
@patch
def aggregate(self: BaseServer):
    raise NotImplementedError("This method should be implemented by subclasses.")
        

### Evaluation




All methods are evaluated using the average of the local evaluation results across all clients.

In [ ]:
#| export
@patch
def evaluate(self: BaseServer, t):
    lst_train_res = []
    lst_test_res = []
    for id in range(self.cfg.num_clients):
        client_state = self.state_mgr.get_state(id)
        client = self.client_fn(client_cls= self.client_cls, id= id, cfg= self.cfg, comm_round= t, client_state= client_state)
        
        res_train = client.evaluate_local(loader= 'train')
        lst_train_res.append(res_train)

        res_test = client.evaluate_local(loader= 'test')
        lst_test_res.append(res_test)
        
        del client
        gc.collect()
        torch.cuda.empty_cache()

    return lst_train_res, lst_test_res    


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()